This notebook will create the UMAPs in Main Figure 4

# Setup

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

sc.settings.verbosity = 3

thisseed = 231103
np.random.seed(thisseed)

In [ ]:
from adjustText import adjust_text

def gen_mpl_labels(
    adata, groupby, exclude=(), ax=None, adjust_kwargs=None, text_kwargs=None, color_by_group=False
):
    if adjust_kwargs is None:
        adjust_kwargs = {"text_from_points": False}
    if text_kwargs is None:
        text_kwargs = {}

    medians = {}

    for g, g_idx in adata.obs.groupby(groupby).groups.items():
        if g in exclude:
            continue
        medians[g] = np.median(adata[g_idx].obsm["X_umap"], axis=0)

    # Fill the text colors dictionary
    text_colors = {group: None for group in adata.obs[groupby].cat.categories}

    if color_by_group and groupby + "_colors" in adata.uns:
        for i, group in enumerate(adata.obs[groupby].cat.categories):
            if group in exclude:
                continue
            text_colors[group] = adata.uns[groupby + "_colors"][i]

    if ax is None:
        texts = [
            plt.text(x=x, y=y, s=k, color=text_colors[k], **text_kwargs) for k, (x, y) in medians.items()
        ]
    else:
        texts = [ax.text(x=x, y=y, s=k, color=text_colors[k], **text_kwargs) for k, (x, y) in medians.items()]

    adjust_text(texts, **adjust_kwargs)

In [ ]:
def gen_mpl_labels_norepel(
    adata, groupby, exclude=(), ax=None, adjust_kwargs=None, text_kwargs=None, color_by_group=False
):
    if adjust_kwargs is None:
        adjust_kwargs = {"text_from_points": False}
    if text_kwargs is None:
        text_kwargs = {}

    medians = {}

    for g, g_idx in adata.obs.groupby(groupby).groups.items():
        if g in exclude:
            continue
        medians[g] = np.median(adata[g_idx].obsm["X_umap"], axis=0)

    # Fill the text colors dictionary
    text_colors = {group: None for group in adata.obs[groupby].cat.categories}

    if color_by_group and groupby + "_colors" in adata.uns:
        for i, group in enumerate(adata.obs[groupby].cat.categories):
            if group in exclude:
                continue
            text_colors[group] = adata.uns[groupby + "_colors"][i]

    if ax is None:
        texts = [
            plt.text(x=x, y=y, s=k, color=text_colors[k], **text_kwargs) for k, (x, y) in medians.items()
        ]
    else:
        texts = [ax.text(x=x, y=y, s=k, color=text_colors[k], **text_kwargs) for k, (x, y) in medians.items()]

# Pre-Post TNF Pathway Score

In [ ]:
sc.set_figure_params(dpi=100)

plt.rcParams.update({'font.weight': 'bold'})
plt.rcParams.update({'axes.labelweight': 'bold'})
plt.rcParams.update({'axes.titleweight': 'bold'})

## Myeloid

In [ ]:
adata_myeloid = sc.read_h5ad('../../predict/data/objects/annotated/predictCOL__CB_myeloid_annotated_251017.h5ad')
adata_myeloid

In [ ]:
mlm_estimate_myeloid = pd.read_csv('../../predict/out/decoupler/progeny/mlm_tnfjak/myeloid__tnfjak_scoring_mlm_estimate.csv')
mlm_estimate_myeloid.set_index('barcode', inplace=True)
mlm_estimate_myeloid.index.name = None
mlm_estimate_myeloid

In [ ]:
colnames = list(mlm_estimate_myeloid.columns)
colnames

In [ ]:
adata_myeloid.obs[colnames] = mlm_estimate_myeloid
adata_myeloid.obs

In [ ]:
adata_myeloid.layers['counts'] = adata_myeloid.X.copy()  # preserve raw counts
adata_myeloid.raw = adata_myeloid.copy()  # set raw for easy recovery after subsetting HVGs
sc.pp.normalize_total(adata_myeloid, target_sum=1e4)
sc.pp.log1p(adata_myeloid)
adata_myeloid

In [ ]:
adata_myeloid_ccd = adata_myeloid[adata_myeloid.obs.diagSub.isin(["cCD"])].copy()
adata_myeloid_ccd

In [ ]:
adata_myeloid_ccd_atnf = adata_myeloid_ccd[adata_myeloid_ccd.obs.antitnf_response.isin(['FR', 'PR', 'NR'])].copy()
adata_myeloid_ccd_atnf

In [ ]:
adata_myeloid_ccd_pre = adata_myeloid_ccd_atnf[adata_myeloid_ccd_atnf.obs.timepoint.isin(["T0"])].copy()
adata_myeloid_ccd_pre

In [ ]:
adata_myeloid_ccd_post = adata_myeloid_ccd_atnf[adata_myeloid_ccd_atnf.obs.timepoint.isin(["R", "R2"])].copy()
adata_myeloid_ccd_post

In [ ]:
with plt.rc_context({}):
    ax = sc.pl.umap(adata_myeloid_ccd_pre, color='TNFa', cmap='turbo', vmin=0, vmax=30, size=20,
                    title='Myeloid TNF Pathway Score\n(cCD Pre-Treatment)',
                    show=False, legend_loc=None, frameon=True)
    gen_mpl_labels(
        adata_myeloid_ccd_pre,
        'cellstate',
        exclude=("None",),  # This was before we had the `nan` behaviour
        ax=ax,
        adjust_kwargs=dict(arrowprops=dict(arrowstyle='-', color='black')),
        text_kwargs=dict(fontsize=12, weight='normal', path_effects=[pe.withStroke(linewidth=1, foreground="darkgray")]),
        color_by_group=False
    )
    fig = ax.get_figure()
    fig.tight_layout()
    plt.show()

In [ ]:
ax = sc.pl.umap(adata_myeloid_ccd_post, color='TNFa', cmap='turbo', vmin=0, vmax=30, size=20,
                title='Myeloid TNF Pathway Score\n(cCD Post-Treatment)',
                show=False, legend_loc=None, frameon=True)
gen_mpl_labels(
    adata_myeloid_ccd_pre,
    'cellstate',
    exclude=("None",),  # This was before we had the `nan` behaviour
    ax=ax,
    adjust_kwargs=dict(arrowprops=dict(arrowstyle='-', color='black')),
    text_kwargs=dict(fontsize=12, weight='normal', path_effects=[pe.withStroke(linewidth=1, foreground="darkgray")]),
    color_by_group=False
)
ax
fig = ax.get_figure()
fig.tight_layout()
plt.show()

In [ ]:
ax = sc.pl.umap(adata_myeloid_ccd_post, color='TNFa', cmap='turbo', vmin=0, vmax=30, size=20,
                title='Myeloid TNF Pathway Score\n(cCD Post-Treatment)',
                show=False, legend_loc=None, frameon=True)
gen_mpl_labels(
    adata_myeloid_ccd_pre,
    'cellstate',
    exclude=("None",),  # This was before we had the `nan` behaviour
    ax=ax,
    adjust_kwargs=dict(arrowprops=dict(arrowstyle='-', color='black')),
    text_kwargs=dict(fontsize=12, weight='normal', path_effects=[pe.withStroke(linewidth=1, foreground="darkgray")]),
    color_by_group=False
)
ax
fig = ax.get_figure()
fig.tight_layout()
plt.show()

In [ ]:
ax = sc.pl.umap(adata_myeloid_ccd_post, color='TNFa', cmap='turbo', vmin=0, vmax=30, size=20,
                title='Myeloid TNF Pathway Score\n(cCD Post-Treatment)',
                show=False, legend_loc=None, frameon=True)
gen_mpl_labels(
    adata_myeloid_ccd_pre,
    'cellstate',
    exclude=("None",),  # This was before we had the `nan` behaviour
    ax=ax,
    adjust_kwargs=dict(arrowprops=dict(arrowstyle='-', color='black')),
    text_kwargs=dict(fontsize=12, weight='normal', path_effects=[pe.withStroke(linewidth=1, foreground="darkgray")]),
    color_by_group=False
)
ax
fig = ax.get_figure()
fig.tight_layout()
plt.show()

In [ ]:
tnf_myeloid_list = ["TisResMac", "Inflm. Mac", "MNP", "Neutrophil", "cDC", "LAMP3+ DC", "Mast Cell"]
cellstate_tnf_myeloid = [x if x in tnf_myeloid_list else None for x in adata_myeloid_ccd_pre.obs.cellstate]

In [ ]:
adata_myeloid_ccd_pre.obs['cellstate_tnf'] = cellstate_tnf_myeloid
adata_myeloid_ccd_pre.obs['cellstate_tnf']

In [ ]:
with plt.rc_context({}):
    ax = sc.pl.umap(adata_myeloid_ccd_pre, color='TNFa', cmap='turbo', vmin=0, vmax=30, size=20,
                    title='Myeloid TNF Pathway Score\n(cCD Pre-Treatment)',
                    show=False, legend_loc=None, frameon=True)
    gen_mpl_labels(
        adata_myeloid_ccd_pre,
        'cellstate_tnf',
        exclude=("None",),  # This was before we had the `nan` behaviour
        ax=ax,
        adjust_kwargs=dict(arrowprops=dict(arrowstyle='-', color='black')),
        text_kwargs=dict(fontsize=10, weight='bold', path_effects=[pe.withStroke(linewidth=1, foreground="darkgray")]),
        color_by_group=False
    )
    fig = ax.get_figure()
    fig.tight_layout()
    plt.show()

In [ ]:
ax = sc.pl.umap(adata_myeloid_ccd_post, color='TNFa', cmap='turbo', vmin=0, vmax=30, size=20,
                title='Myeloid TNF Pathway Score\n(cCD Post-Treatment)',
                show=False, legend_loc=None, frameon=True)
gen_mpl_labels(
    adata_myeloid_ccd_pre,
    'cellstate_tnf',
    exclude=("None",),  # This was before we had the `nan` behaviour
    ax=ax,
    adjust_kwargs=dict(arrowprops=dict(arrowstyle='-', color='black')),
    text_kwargs=dict(fontsize=10, weight='bold', path_effects=[pe.withStroke(linewidth=1, foreground="darkgray")]),
    color_by_group=False
)
ax
fig = ax.get_figure()
fig.tight_layout()
plt.show()

## TNKILC

In [ ]:
adata_tnkilc = sc.read_h5ad('../../predict/data/objects/annotated/predictCOL__CB_tnkilc_annotated_251017.h5ad')
adata_tnkilc

In [ ]:
mlm_estimate_tnkilc = pd.read_csv('../../predict/out/decoupler/progeny/mlm_tnfjak/tnkilc__tnfjak_scoring_mlm_estimate.csv')
mlm_estimate_tnkilc.set_index('barcode', inplace=True)
mlm_estimate_tnkilc.index.name = None
mlm_estimate_tnkilc

In [ ]:
colnames = list(mlm_estimate_tnkilc.columns)
colnames

In [ ]:
adata_tnkilc.obs[colnames] = mlm_estimate_tnkilc
adata_tnkilc.obs

In [ ]:
adata_tnkilc.layers['counts'] = adata_tnkilc.X.copy()  # preserve raw counts
adata_tnkilc.raw = adata_tnkilc.copy()  # set raw for easy recovery after subsetting HVGs
sc.pp.normalize_total(adata_tnkilc, target_sum=1e4)
sc.pp.log1p(adata_tnkilc)
adata_tnkilc

In [ ]:
adata_tnkilc_ccd = adata_tnkilc[adata_tnkilc.obs.diagSub.isin(["cCD"])].copy()
adata_tnkilc_ccd

In [ ]:
adata_tnkilc_ccd_atnf = adata_tnkilc_ccd[adata_tnkilc_ccd.obs.antitnf_response.isin(['FR', 'PR', 'NR'])].copy()
adata_tnkilc_ccd_atnf

In [ ]:
adata_tnkilc_ccd_pre = adata_tnkilc_ccd_atnf[adata_tnkilc_ccd_atnf.obs.timepoint.isin(["T0"])].copy()
adata_tnkilc_ccd_pre

In [ ]:
adata_tnkilc_ccd_post = adata_tnkilc_ccd_atnf[adata_tnkilc_ccd_atnf.obs.timepoint.isin(["R", "R2"])].copy()
adata_tnkilc_ccd_post

In [ ]:
with plt.rc_context({}):
    ax = sc.pl.umap(adata_tnkilc_ccd_pre, color='TNFa', cmap='turbo', vmin=0, vmax=20, size=20,
                    title='TNKILC TNF Pathway Score\n(cCD Pre-Treatment)',
                    show=False, legend_loc=None, frameon=True)
    gen_mpl_labels(
        adata_tnkilc_ccd_pre,
        'cellstate',
        exclude=("None",),  # This was before we had the `nan` behaviour
        ax=ax,
        adjust_kwargs=dict(arrowprops=dict(arrowstyle='-', color='black')),
        text_kwargs=dict(fontsize=12, weight='normal', path_effects=[pe.withStroke(linewidth=1, foreground="darkgray")]),
        color_by_group=False
    )
    fig = ax.get_figure()
    fig.tight_layout()
    plt.show()

In [ ]:
with plt.rc_context({}):
    ax = sc.pl.umap(adata_tnkilc_ccd_post, color='TNFa', cmap='turbo', vmin=0, vmax=20, size=20,
                    title='TNKILC TNF Pathway Score\n(cCD Post-Treatment)',
                    show=False, legend_loc=None, frameon=True)
    gen_mpl_labels(
        adata_tnkilc_ccd_pre,
        'cellstate',
        exclude=("None",),  # This was before we had the `nan` behaviour
        ax=ax,
        adjust_kwargs=dict(arrowprops=dict(arrowstyle='-', color='black')),
        text_kwargs=dict(fontsize=12, weight='normal', path_effects=[pe.withStroke(linewidth=1, foreground="darkgray")]),
        color_by_group=False
    )
    fig = ax.get_figure()
    fig.tight_layout()
    plt.show()

In [ ]:
tnf_tnkilc_list = ["Th1", "Th17", "Treg", "MemoryCD4", "MAIT", "XCL+ NK", "ILC"]
cellstate_tnf_tnkilc = [x if x in tnf_tnkilc_list else None for x in adata_tnkilc_ccd_pre.obs.cellstate]

In [ ]:
adata_tnkilc_ccd_pre.obs['cellstate_tnf'] = cellstate_tnf_tnkilc
adata_tnkilc_ccd_pre.obs['cellstate_tnf']

In [ ]:
with plt.rc_context({}):
    ax = sc.pl.umap(adata_tnkilc_ccd_pre, color='TNFa', cmap='turbo', vmin=0, vmax=20, size=20,
                    title='TNKILC TNF Pathway Score\n(cCD Pre-Treatment)',
                    show=False, legend_loc=None, frameon=True)
    gen_mpl_labels(
        adata_tnkilc_ccd_pre,
        'cellstate_tnf',
        exclude=("None",),  # This was before we had the `nan` behaviour
        ax=ax,
        adjust_kwargs=dict(arrowprops=dict(arrowstyle='-', color='black')),
        text_kwargs=dict(fontsize=12, weight='bold', path_effects=[pe.withStroke(linewidth=1, foreground="darkgray")]),
        color_by_group=False
    )
    fig = ax.get_figure()
    fig.tight_layout()
    plt.show()

In [ ]:
with plt.rc_context({}):
    ax = sc.pl.umap(adata_tnkilc_ccd_post, color='TNFa', cmap='turbo', vmin=0, vmax=20, size=20,
                    title='TNKILC TNF Pathway Score\n(cCD Post-Treatment)',
                    show=False, legend_loc=None, frameon=True)
    gen_mpl_labels(
        adata_tnkilc_ccd_pre,
        'cellstate_tnf',
        exclude=("None",),  # This was before we had the `nan` behaviour
        ax=ax,
        adjust_kwargs=dict(arrowprops=dict(arrowstyle='-', color='black')),
        text_kwargs=dict(fontsize=12, weight='bold', path_effects=[pe.withStroke(linewidth=1, foreground="darkgray")]),
        color_by_group=False
    )
    fig = ax.get_figure()
    fig.tight_layout()
    plt.show()